# MAPseq quality control

**Audience:** MAPseq analysts reviewing whether a processed run is trustworthy.

**Learning goals:** reconcile filtering totals, inspect RT-index rescue, distinguish input observations from derived representatives, and identify configuration problems before biological interpretation.

## Outline

1. Ensure the synthetic example pipeline has run.
2. Inspect read-filter counts and retention.
3. Inspect sample-index mapping status.
4. Reconcile reads across every processing layer.
5. Review practical QC warnings.

In [ ]:
from __future__ import annotations

import sqlite3
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
database = ROOT / 'examples' / 'output' / 'example.sqlite'
if not database.exists():
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'generate_synthetic_mapseq.py')], check=True)
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'run_example_pipeline.py')], check=True)
con = sqlite3.connect(database)
database

## Read filtering separates failure modes

A useful QC report shows why reads were removed instead of reporting one aggregate loss. The synthetic data contains known constant, quality, barcode-N, and UMI-N failures.

In [ ]:
qc = dict(con.execute('SELECT key, value FROM qc'))
removed = {key: value for key, value in qc.items() if key not in {'total_reads', 'retained_reads'}}
retention = qc['retained_reads'] / qc['total_reads']
print('QC:', qc)
print(f'Retention: {retention:.1%}')
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(list(removed), list(removed.values()), color='#3D8DFF')
ax.set(title='Reads removed by QC category', ylabel='Read pairs')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

## RT-index correction remains auditable

Exact and uniquely corrected indices are assigned to known samples. Ambiguous and unassigned strings are retained rather than forced.

In [ ]:
index_status = con.execute('''
SELECT m.status, count(*) AS observed_strings, sum(r.read_count) AS reads
FROM sample_index_mapping m
JOIN raw_counts r ON r.sample_index = m.observed_index
GROUP BY m.status ORDER BY reads DESC
''').fetchall()
index_examples = con.execute('SELECT observed_index, sample_index, distance, status FROM sample_index_mapping ORDER BY status, observed_index').fetchall()
print('Status summary:', index_status)
print('Mappings:', index_examples)

## Read conservation catches join or aggregation errors

With no pre-collapse support filter, read totals must agree across raw, barcode-collapsed, UMI-collapsed, and molecule-summary tables.

In [ ]:
totals = {table: con.execute(f'SELECT sum(read_count) FROM {table}').fetchone()[0] for table in ['raw_counts', 'collapsed_counts', 'umi_collapsed_counts', 'molecule_counts']}
coverage = {
    'barcode mappings missing': con.execute('SELECT count(*) FROM barcode_scores s LEFT JOIN barcode_mapping m USING(barcode) WHERE m.barcode IS NULL').fetchone()[0],
    'UMI mappings missing': con.execute('SELECT count(*) FROM collapsed_counts c LEFT JOIN umi_mapping u USING(barcode, sample_index, umi) WHERE u.umi IS NULL').fetchone()[0],
}
con.close()
print('Read totals:', totals)
print('Mapping coverage:', coverage)
assert len(set(totals.values())) == 1
assert not any(coverage.values())

## Exercise and interpretation

**Exercise:** change the assumed R2 UMI length by one base and predict how the index-status summary changes. Do this only on a copied database.

**Warning signs:** high constant mismatch suggests the wrong R1 layout; many unassigned indices suggest the wrong R2 split/orientation or whitelist; non-conserved reads suggest an explicit support filter or a faulty join.

**Extension:** add per-cycle quality summaries upstream if base-quality degradation is position-specific.